# 🛒 ReviewInsight AI – klientų atsiliepimų analizės sistema e. parduotuvei

**Dalykas:** _Gilusis mokymas naudojant TensorFlow_ · **Darbas:** 1.2.2 – DI sistemos prototipas „Google Colab“ aplinkoje
**Autoriai:** _Rūta Stunžėnaitė, Justas Antanaitis DIEf-24_ · **GitHub:** _https://github.com/ruta-stunzenaite/ReviewInsight-AI.git_

---

## 1. Sistemos apžvalga

E. parduotuvė kasdien gauna daug klientų atsiliepimų. **ReviewInsight AI** juos automatiškai analizuoja
generatyviuoju DI (**Google Gemini API**) ir padeda darbuotojams greičiau reaguoti į problemas.

Kiekvienam atsiliepimui sistema grąžina **struktūrizuotą JSON**:

| Laukas | Reikšmė |
|---|---|
| `sentiment` + `confidence` | nuotaika (positive / neutral / negative / mixed) ir pasitikėjimo balas 0–1 |
| `categories` | problemų kategorijos iš fiksuoto sąrašo |
| `summary` | trumpa santrauka lietuvių kalba |
| `urgency` | skubumas: low / medium / high / critical |
| `reply_draft` | atsakymo klientui juodraštis ta pačia kalba |

Sistema veikia pagal **human-in-the-loop** principą – klientas negauna jokio DI atsakymo, kol jo nepatvirtina vadybininkas.

**Technologijos:** Python · `google-genai` (Gemini API) · pandas · matplotlib · SQLite · ipywidgets

## 2. Apdorojimo konvejeris (nuo įvesties iki išvesties)

```
 CSV/JSON failas ──► [1] ReviewLoader ──► [2] ReviewPreprocessor ──► [3] GeminiAnalyzer ⇄ Gemini API
                     (įkėlimas,           (valymas, kalba,           (prompt + JSON schema,
                      stulpelių patikra)   per trumpų atmetimas)       pakartojimai, backoff)
                                                                            │
                                                                            ▼
 [7] Įvertinimas su y ◄── [6] ReportGenerator ◄── [5] ReviewApprovalService ◄── [4] ResponseValidator
     (data/y/)               (diagramos)            (vadybininko sprendimas)        (JSON patikra)
                                         ▲
                                  ReviewRepository (SQLite)
```

## 3. Aplinkos paruošimas

**Prieš paleidžiant:**
1. Gaukite API raktą [Google AI Studio](https://aistudio.google.com/apikey).
2. Colab kairėje paspauskite 🔑 **Secrets** → pridėkite `GOOGLE_API_KEY` ir įjunkite „Notebook access“.

In [1]:
#@title 🔧 Priklausomybių diegimas
!pip install -q google-genai langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 16.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [10]:
#@title 🔑 Gemini API kliento konfigūravimas
import os, json, re, time, sqlite3, textwrap
from dataclasses import dataclass, field, asdict
from datetime import datetime
from enum import Enum
from typing import List, Optional

import pandas as pd
import matplotlib.pyplot as plt
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception as e:
    print("⚠️ Colab Secrets nepasiekiami – naudojamas GEMINI_API_KEY iš aplinkos.", e)

client = genai.Client()  # raktas paimamas iš GEMINI_API_KEY

# Modeliai bandomi iš eilės: jei pirmasis nepasiekiamas jūsų paskyrai, imamas kitas.
MODEL_CANDIDATES = ["gemini-3.8-flash", "gemini-3.5-flash-lite", "gemini-flash-latest", "gemini-2.5-flash"]
TEMPERATURE = 0.2
print("✅ Gemini klientas paruoštas. Kandidatiniai modeliai:", MODEL_CANDIDATES)

✅ Gemini klientas paruoštas. Kandidatiniai modeliai: ['gemini-3.8-flash', 'gemini-3.5-flash-lite', 'gemini-flash-latest', 'gemini-2.5-flash']


## Žingsnis 1 – įvesties duomenų priėmimas (`ReviewLoader`)

Įvestis – iš e. parduotuvės eksportuotas CSV failas (`1.2.2_Colab_prototipas/input_data/reviews_sample.csv` GitHub repozitorijoje).
Kiekviena eilutė yra viena **X** įvestis; atitinkami laukiami rezultatai **y** saugomi faile `1.2.2_Colab_prototipas/input_data/expected_outputs.json`.

Tai tie patys duomenys kaip `1.2.1_E2E_pavyzdziai/data/X/` ir `data/y/` porose, tik sujungti į du failus.

Užrašinė pirmiausia bando atsisiųsti duomenis iš GitHub. Jei nuoroda dar nenurodyta, naudojama įterpta tų pačių duomenų kopija – todėl užrašinė veikia bet kuriuo atveju.

In [6]:
#@title 📥 Duomenų šaltinis
# Įvesties duomenys (X) ir laukiami rezultatai (y) iš GitHub repozitorijos:
GITHUB_RAW = "https://raw.githubusercontent.com/ruta-stunzenaite/ReviewInsight-AI/refs/heads/main/1.2.2_Colab_prototipas/input_data"

EMBEDDED_X = [
 {"review_id":"R001","product_id":"P-1023","product_name":"Belaidės ausinės SoundAir","rating":5,"date":"2026-09-28","text":"Ausinės puikios! Garsas aiškus, baterija laiko visą dieną, o pristatė vos per dvi dienas. Tikrai rekomenduoju."},
 {"review_id":"R002","product_id":"P-2210","product_name":"Kavos aparatas BaristaPro","rating":1,"date":"2026-09-29","text":"Užsakymo laukiau 3 savaites, nors buvo žadėta per 3 darbo dienas. Dėžė atvyko sulamdyta, aparatas neįsijungia. Į el. laiškus niekas neatsako. Noriu grąžinti pinigus!"},
 {"review_id":"R003","product_id":"P-0871","product_name":"Dulkių siurblys CleanMax 300","rating":3,"date":"2026-09-30","text":"Siurblys traukia gerai, bet jis daug triukšmingesnis, nei nurodyta aprašyme, o kaina už tokį modelį per didelė."},
 {"review_id":"R004","product_id":"P-3305","product_name":"Elektrinis virdulys HeatUp","rating":1,"date":"2026-10-01","text":"Po savaitės naudojimo virdulio laidas ties jungtimi įkaito ir pradėjo kvepėti degėsiais. Tai pavojinga, bijau juo naudotis!"},
 {"review_id":"R005","product_id":"P-1023","product_name":"Belaidės ausinės SoundAir","rating":4,"date":"2026-10-02","text":"Good headphones with great sound, but the charging case feels cheap and the lid is loose. Delivery was fast."},
 {"review_id":"R006","product_id":"P-2210","product_name":"Kavos aparatas BaristaPro","rating":5,"date":"2026-10-03","text":"Super!!!"},
]
EMBEDDED_Y = [
 {"review_id":"R001","status":"ANALYZED","language":"lt","sentiment":"positive","urgency":"low","categories":["produkto_kokybe","pristatymas"]},
 {"review_id":"R002","status":"ANALYZED","language":"lt","sentiment":"negative","urgency":"high","categories":["pristatymas","pakuote","produkto_kokybe","klientu_aptarnavimas","grazinimas_ir_garantija"]},
 {"review_id":"R003","status":"ANALYZED","language":"lt","sentiment":"mixed","urgency":"medium","categories":["produkto_aprasymas","kaina","produkto_kokybe"]},
 {"review_id":"R004","status":"ANALYZED","language":"lt","sentiment":"negative","urgency":"critical","categories":["saugumas","produkto_kokybe"]},
 {"review_id":"R005","status":"ANALYZED","language":"en","sentiment":"mixed","urgency":"low","categories":["produkto_kokybe","pristatymas"]},
 {"review_id":"R006","status":"REJECTED","language":None,"sentiment":None,"urgency":None,"categories":[]},
]

os.makedirs("data", exist_ok=True)
INPUT_PATH, EXPECTED_PATH = "data/reviews_sample.csv", "data/expected_outputs.json"
try:
    if "USERNAME" in GITHUB_RAW:
        raise RuntimeError("GitHub nuoroda dar nenurodyta")
    pd.read_csv(f"{GITHUB_RAW}/reviews_sample.csv").to_csv(INPUT_PATH, index=False)
    pd.read_json(f"{GITHUB_RAW}/expected_outputs.json").to_json(EXPECTED_PATH, orient="records", force_ascii=False)
    print("✅ Duomenys atsisiųsti iš GitHub")
except Exception as e:
    pd.DataFrame(EMBEDDED_X).to_csv(INPUT_PATH, index=False)
    json.dump(EMBEDDED_Y, open(EXPECTED_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print(f"ℹ️ Naudojama įterpta duomenų kopija ({e})")

✅ Duomenys atsisiųsti iš GitHub


In [7]:
#@title 🧱 Domeno modelis (Review, AnalysisResult, enum tipai)
class ReviewStatus(str, Enum):
    NEW = "NEW"; PREPROCESSED = "PREPROCESSED"; REJECTED = "REJECTED"
    ANALYZED = "ANALYZED"; ANALYSIS_FAILED = "ANALYSIS_FAILED"

class Sentiment(str, Enum):
    positive = "positive"; neutral = "neutral"; negative = "negative"; mixed = "mixed"

class UrgencyLevel(str, Enum):
    low = "low"; medium = "medium"; high = "high"; critical = "critical"

class DecisionType(str, Enum):
    APPROVED = "APPROVED"; EDITED = "EDITED"; REJECTED = "REJECTED"

ALLOWED_CATEGORIES = ["produkto_kokybe", "pristatymas", "pakuote", "kaina", "klientu_aptarnavimas",
                      "produkto_aprasymas", "grazinimas_ir_garantija", "saugumas", "kita"]

@dataclass
class Review:
    review_id: str
    product_id: str
    product_name: str
    rating: int
    date: str
    text: str
    clean_text: str = ""
    language: Optional[str] = None
    status: ReviewStatus = ReviewStatus.NEW
    reject_reason: Optional[str] = None

@dataclass
class AnalysisResult:
    review_id: str
    sentiment: str
    confidence: float
    categories: List[str]
    summary: str
    urgency: str
    reply_draft: str
    model_name: str = ""
    prompt_version: int = 1
    attempts: int = 1
    created_at: str = field(default_factory=lambda: datetime.now().isoformat(timespec="seconds"))

    @staticmethod
    def from_json(data: dict, review_id: str, model_name: str, attempts: int) -> "AnalysisResult":
        return AnalysisResult(review_id=review_id, model_name=model_name, attempts=attempts, **data)

    def needs_attention(self) -> bool:
        return self.urgency in ("high", "critical") or self.confidence < 0.6

class AnalysisError(Exception):
    def __init__(self, review_id, attempts, last_error):
        super().__init__(f"{review_id}: nepavyko po {attempts} bandymų – {last_error}")
        self.review_id, self.attempts, self.last_error = review_id, attempts, last_error
print("✅ Domeno klasės apibrėžtos")

✅ Domeno klasės apibrėžtos


In [8]:
#@title 📂 ReviewLoader – failo įkėlimas ir stulpelių patikra
class ReviewLoader:
    required_columns = ["review_id", "product_id", "product_name", "rating", "date", "text"]

    def load_file(self, file_path: str) -> pd.DataFrame:
        df = pd.read_json(file_path) if file_path.endswith(".json") else pd.read_csv(file_path)
        self.validate_columns(df)
        return df

    def validate_columns(self, df: pd.DataFrame) -> bool:
        missing = [c for c in self.required_columns if c not in df.columns]
        if missing:
            raise ValueError(f"Faile trūksta stulpelių: {missing}")
        return True

    def to_reviews(self, df: pd.DataFrame) -> List[Review]:
        return [Review(**{c: row[c] for c in self.required_columns}) for _, row in df.iterrows()]

loader = ReviewLoader()
df_input = loader.load_file(INPUT_PATH)
reviews = loader.to_reviews(df_input)
print(f"✅ Įkelta atsiliepimų: {len(reviews)}")
df_input

✅ Įkelta atsiliepimų: 6


,review_id,product_id,product_name,rating,date,text
0,R001,P-1023,Belaidės ausinės SoundAir,5,2026-09-28,"Ausinės puikios! Garsas aiškus, baterija laiko..."
1,R002,P-2210,Kavos aparatas BaristaPro,1,2026-09-29,"Užsakymo laukiau 3 savaites, nors buvo žadėta ..."
2,R003,P-0871,Dulkių siurblys CleanMax 300,3,2026-09-30,"Siurblys traukia gerai, bet jis daug triukšmin..."
3,R004,P-3305,Elektrinis virdulys HeatUp,1,2026-10-01,Po savaitės naudojimo virdulio laidas ties jun...
4,R005,P-1023,Belaidės ausinės SoundAir,4,2026-10-02,"Good headphones with great sound, but the char..."
5,R006,P-2210,Kavos aparatas BaristaPro,5,2026-10-03,Super!!!


## Žingsnis 2 – teksto paruošimas (`ReviewPreprocessor`)

* pašalinami HTML likučiai, nereikalingi tarpai ir pasikartojantys skyrybos ženklai;
* nustatoma kalba (`langdetect`, o jei jis neveikia – paprasta euristika pagal lietuviškas raides);
* atsiliepimai, kurių tekstas be skyrybos trumpesnis nei **15 simbolių**, atmetami (`REJECTED`) ir **į Gemini API nesiunčiami** – taip taupomos užklausos.

In [9]:
#@title 🧹 ReviewPreprocessor
class ReviewPreprocessor:
    def __init__(self, min_length: int = 15, supported_languages=("lt", "en")):
        self.min_length = min_length
        self.supported_languages = supported_languages

    def clean_text(self, text: str) -> str:
        text = re.sub(r"<[^>]+>", " ", str(text))
        text = re.sub(r"([!?.])\1+", r"\1", text)
        return re.sub(r"\s+", " ", text).strip()

    def detect_language(self, text: str) -> str:
        try:
            from langdetect import detect, DetectorFactory
            DetectorFactory.seed = 0
            lang = detect(text)
        except Exception:
            lang = "lt" if re.search(r"[ąčęėįšųūž]", text.lower()) else "en"
        return lang if lang in self.supported_languages else "kita"

    def is_too_short(self, text: str) -> bool:
        return len(re.sub(r"[^\w]", "", text)) < self.min_length

    def process(self, reviews: List[Review]) -> List[Review]:
        for r in reviews:
            r.clean_text = self.clean_text(r.text)
            if self.is_too_short(r.clean_text):
                r.status, r.reject_reason = ReviewStatus.REJECTED, "per trumpas tekstas"
                continue
            r.language = self.detect_language(r.clean_text)
            r.status = ReviewStatus.PREPROCESSED
        return reviews

preprocessor = ReviewPreprocessor()
reviews = preprocessor.process(reviews)
pd.DataFrame([{"review_id": r.review_id, "kalba": r.language, "statusas": r.status.value,
               "priežastis": r.reject_reason, "išvalytas tekstas": r.clean_text} for r in reviews])

,review_id,kalba,statusas,priežastis,išvalytas tekstas
0,R001,lt,PREPROCESSED,None,"Ausinės puikios! Garsas aiškus, baterija laiko..."
1,R002,lt,PREPROCESSED,None,"Užsakymo laukiau 3 savaites, nors buvo žadėta ..."
2,R003,lt,PREPROCESSED,None,"Siurblys traukia gerai, bet jis daug triukšmin..."
3,R004,lt,PREPROCESSED,None,Po savaitės naudojimo virdulio laidas ties jun...
4,R005,en,PREPROCESSED,None,"Good headphones with great sound, but the char..."
5,R006,None,REJECTED,per trumpas tekstas,Super!


## Žingsnis 3 – prompt'as ir struktūrizuoto atsakymo schema

Paduodamas prompt'o šablonas. Gemini API iškvietime nurodomi
`response_mime_type="application/json"` ir `response_schema`, todėl modelis grąžina JSON pagal schemą.

In [10]:
#@title 📝 Prompt šablonas ir JSON schema
PROMPT_VERSION = 1
PROMPT_TEMPLATE = textwrap.dedent("""
Tu esi e. parduotuvės klientų aptarnavimo analitikas. Išanalizuok vieną kliento atsiliepimą
ir grąžink TIK JSON objektą pagal nurodytą schemą.

TAISYKLĖS:
1. sentiment – viena iš: positive, neutral, negative, mixed.
   „mixed“ naudok, kai atsiliepime yra ir aiškiai teigiamų, ir aiškiai neigiamų aspektų.
2. confidence – tavo pasitikėjimas nuotaikos įvertinimu, skaičius nuo 0 iki 1.
3. categories – 1–5 problemų / temų kategorijos TIK iš šio sąrašo:
   {allowed_categories}
4. summary – viena glausta santrauka lietuvių kalba (iki 30 žodžių).
5. urgency – viena iš: low, medium, high, critical.
   - critical: bet kokia saugumo ar sveikatos rizika (įkaitimas, dūmai, elektra, sužalojimas);
   - high: neveikianti prekė, grąžinimo / pinigų reikalavimas, keli rimti nusiskundimai;
   - medium: dalinis nepasitenkinimas, neatitikimas aprašymui, kainos skundas;
   - low: teigiamas arba nedidelės pastabos.
6. reply_draft – mandagus atsakymo klientui juodraštis (2–4 sakiniai) TA PAČIA KALBA,
   kuria parašytas atsiliepimas. Nežadėk nieko, ko parduotuvė negali užtikrinti, išskyrus
   grąžinimą per 14 dienų ir garantinį aptarnavimą. Saugumo atveju pirmiausia patark nebenaudoti prekės.

ATSILIEPIMAS:
- Prekė: {product_name} ({product_id})
- Įvertinimas: {rating}/5
- Kalba: {language}
- Tekstas: \"\"\"{text}\"\"\"
""").strip()

RESPONSE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "sentiment":   {"type": "STRING", "enum": [s.value for s in Sentiment]},
        "confidence":  {"type": "NUMBER"},
        "categories":  {"type": "ARRAY", "items": {"type": "STRING", "enum": ALLOWED_CATEGORIES}},
        "summary":     {"type": "STRING"},
        "urgency":     {"type": "STRING", "enum": [u.value for u in UrgencyLevel]},
        "reply_draft": {"type": "STRING"},
    },
    "required": ["sentiment", "confidence", "categories", "summary", "urgency", "reply_draft"],
}

def build_prompt(review: Review, errors: Optional[List[str]] = None) -> str:
    prompt = PROMPT_TEMPLATE.format(allowed_categories=", ".join(ALLOWED_CATEGORIES),
                                    product_name=review.product_name, product_id=review.product_id,
                                    rating=review.rating, language=review.language, text=review.clean_text)
    if errors:
        prompt += "\n\nANKSTESNIS TAVO ATSAKYMAS BUVO NEVALIDUS. Klaidos:\n- " + "\n- ".join(errors) \
                  + "\nPataisyk ir grąžink tik validų JSON."
    return prompt

print(build_prompt(reviews[0]))

Tu esi e. parduotuvės klientų aptarnavimo analitikas. Išanalizuok vieną kliento atsiliepimą
ir grąžink TIK JSON objektą pagal nurodytą schemą.

TAISYKLĖS:
1. sentiment – viena iš: positive, neutral, negative, mixed.
   „mixed“ naudok, kai atsiliepime yra ir aiškiai teigiamų, ir aiškiai neigiamų aspektų.
2. confidence – tavo pasitikėjimas nuotaikos įvertinimu, skaičius nuo 0 iki 1.
3. categories – 1–5 problemų / temų kategorijos TIK iš šio sąrašo:
   produkto_kokybe, pristatymas, pakuote, kaina, klientu_aptarnavimas, produkto_aprasymas, grazinimas_ir_garantija, saugumas, kita
4. summary – viena glausta santrauka lietuvių kalba (iki 30 žodžių).
5. urgency – viena iš: low, medium, high, critical.
   - critical: bet kokia saugumo ar sveikatos rizika (įkaitimas, dūmai, elektra, sužalojimas);
   - high: neveikianti prekė, grąžinimo / pinigų reikalavimas, keli rimti nusiskundimai;
   - medium: dalinis nepasitenkinimas, neatitikimas aprašymui, kainos skundas;
   - low: teigiamas arba nedidelės

## Žingsnis 4 – atsakymo validavimas (`ResponseValidator`)

Net naudojant JSON schemą, DI atsakymas tikrinamas papildomai: ar JSON korektiškas, ar yra visi laukai,
ar reikšmės leidžiamos, ar `confidence` intervale [0, 1], ar tekstai netušti.

In [11]:
#@title ✅ ResponseValidator
class ResponseValidator:
    def __init__(self, allowed_categories=ALLOWED_CATEGORIES):
        self.allowed_categories = allowed_categories
        self.errors: List[str] = []

    def parse(self, raw_json: str) -> dict:
        raw = raw_json.strip()
        raw = re.sub(r"^```(?:json)?|```$", "", raw).strip()   # jei modelis apgaubė ```json
        return json.loads(raw)

    def validate(self, raw_json: str) -> bool:
        self.errors = []
        try:
            data = self.parse(raw_json)
        except Exception as e:
            self.errors.append(f"neteisingas JSON: {e}")
            return False
        for key in RESPONSE_SCHEMA["required"]:
            if key not in data:
                self.errors.append(f"trūksta lauko '{key}'")
        if self.errors:
            return False
        if data["sentiment"] not in [s.value for s in Sentiment]:
            self.errors.append(f"neleistina sentiment reikšmė '{data['sentiment']}'")
        if data["urgency"] not in [u.value for u in UrgencyLevel]:
            self.errors.append(f"neleistina urgency reikšmė '{data['urgency']}'")
        if not isinstance(data["confidence"], (int, float)) or not 0 <= data["confidence"] <= 1:
            self.errors.append("confidence turi būti skaičius nuo 0 iki 1")
        bad = [c for c in data["categories"] if c not in self.allowed_categories]
        if bad or not 1 <= len(data["categories"]) <= 5:
            self.errors.append(f"kategorijos turi būti 1–5 iš leidžiamo sąrašo (neleistinos: {bad})")
        for key in ("summary", "reply_draft"):
            if not str(data[key]).strip():
                self.errors.append(f"laukas '{key}' tuščias")
        return not self.errors

    def get_errors(self) -> List[str]:
        return self.errors

# Greitas patikrinimas su sąmoningai klaidingu atsakymu:
v = ResponseValidator()
print(v.validate('{"sentiment": "angry", "confidence": 1.4, "categories": ["oras"], "summary": "x", "urgency": "low", "reply_draft": "y"}'))
print(v.get_errors())

False
["neleistina sentiment reikšmė 'angry'", 'confidence turi būti skaičius nuo 0 iki 1', "kategorijos turi būti 1–5 iš leidžiamo sąrašo (neleistinos: ['oras'])"]


## Žingsnis 5 – DI analizė per Gemini API (`GeminiAnalyzer`)

Patikimumo mechanizmas (atitinka sekų diagramos `loop` / `alt` blokus):
* **nevalidus JSON** → klaidos įrašomos į prompt'ą ir užklausa kartojama (modelis pats pasitaiso);
* **API klaida** (pvz., 429 – viršytas limitas, 503) → eksponentinis laukimas 2, 4, 8 s;
* **modelis nepasiekiamas** (404) → imamas kitas modelis iš `MODEL_CANDIDATES`;
* viršijus `max_retries` → `AnalysisError`, atsiliepimas pažymimas `ANALYSIS_FAILED`, bet kiti apdorojami toliau.

In [12]:
#@title 🤖 GeminiAnalyzer
class GeminiAnalyzer:
    def __init__(self, client, model_candidates=MODEL_CANDIDATES, temperature=TEMPERATURE, max_retries=3):
        self.client = client
        self.model_candidates = list(model_candidates)
        self.model_name = self.model_candidates[0]
        self.temperature = temperature
        self.max_retries = max_retries
        self.validator = ResponseValidator()
        self.log: List[dict] = []   # klaidų žurnalas

    def call_api(self, prompt: str) -> str:
        while True:
            try:
                response = self.client.models.generate_content(
                    model=self.model_name,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        temperature=self.temperature,
                        response_mime_type="application/json",
                        response_schema=RESPONSE_SCHEMA,
                        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),  # 🆕 be AFC įspėjimo
                    ),
                )
                return response.text
            except Exception as e:
                msg = str(e)
                if ("404" in msg or "NOT_FOUND" in msg) and len(self.model_candidates) > 1:
                    self.model_candidates.pop(0)
                    print(f"   ⚠️ modelis {self.model_name} nepasiekiamas → bandomas {self.model_candidates[0]}")
                    self.model_name = self.model_candidates[0]
                    continue
                raise

    # 🆕 po 429 laukiama ilgiau (10, 20, 40 s), po kitų klaidų – 2, 4, 8 s
    def backoff(self, attempt: int, rate_limited: bool = False) -> None:
        wait = (10 if rate_limited else 2) * 2 ** (attempt - 1)
        print(f"   ⏳ laukiama {wait} s prieš bandymą #{attempt + 1}")
        time.sleep(wait)

    def analyze(self, review: Review) -> AnalysisResult:
        errors, last_error = None, ""
        for attempt in range(1, self.max_retries + 1):
            try:
                raw = self.call_api(build_prompt(review, errors))
            except Exception as e:                         # alt: API klaida
                last_error = f"API klaida: {str(e)[:150]}"
                self.log.append({"review_id": review.review_id, "attempt": attempt, "error": last_error})
                if attempt < self.max_retries:             # 🆕 po paskutinio bandymo nelaukiama
                    self.backoff(attempt, rate_limited=("429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)))
                continue
            if self.validator.validate(raw):               # alt: validus JSON
                data = self.validator.parse(raw)
                return AnalysisResult.from_json(data, review.review_id, self.model_name, attempt)
            errors = self.validator.get_errors()           # alt: nevalidus JSON
            last_error = "; ".join(errors)
            self.log.append({"review_id": review.review_id, "attempt": attempt, "error": last_error})
        raise AnalysisError(review.review_id, self.max_retries, last_error)

analyzer = GeminiAnalyzer(client)
print("✅ GeminiAnalyzer paruoštas")

✅ GeminiAnalyzer paruoštas


In [13]:
#@title 🗄️ ReviewRepository – SQLite saugykla
class ReviewRepository:
    def __init__(self, db_path: str = "reviewinsight.db"):
        self.db_path = db_path
        if os.path.exists(db_path):
            os.remove(db_path)
        self.connection = sqlite3.connect(db_path)
        self.connection.executescript("""
        CREATE TABLE reviews (review_id TEXT PRIMARY KEY, product_id TEXT, product_name TEXT, rating INT,
                              date TEXT, text TEXT, language TEXT, status TEXT, reject_reason TEXT);
        CREATE TABLE analysis (review_id TEXT PRIMARY KEY, sentiment TEXT, confidence REAL, categories TEXT,
                               summary TEXT, urgency TEXT, reply_draft TEXT, model_name TEXT,
                               prompt_version INT, attempts INT, created_at TEXT);
        CREATE TABLE decisions (review_id TEXT PRIMARY KEY, decision TEXT, final_reply TEXT,
                                manager_name TEXT, comment TEXT, decided_at TEXT);
        """)

    def save_reviews(self, reviews: List[Review]) -> None:
        self.connection.executemany("INSERT OR REPLACE INTO reviews VALUES (?,?,?,?,?,?,?,?,?)",
            [(r.review_id, r.product_id, r.product_name, int(r.rating), r.date, r.text, r.language,
              r.status.value, r.reject_reason) for r in reviews])
        self.connection.commit()

    def update_status(self, review_id: str, status: ReviewStatus) -> None:
        self.connection.execute("UPDATE reviews SET status=? WHERE review_id=?", (status.value, review_id))
        self.connection.commit()

    def save_analysis(self, a: AnalysisResult) -> None:
        d = asdict(a); d["categories"] = json.dumps(a.categories, ensure_ascii=False)
        self.connection.execute(f"INSERT OR REPLACE INTO analysis VALUES ({','.join('?'*len(d))})", list(d.values()))
        self.connection.commit()

    def save_decision(self, review_id, decision: DecisionType, final_reply, manager_name="vadybininkas", comment=""):
        self.connection.execute("INSERT OR REPLACE INTO decisions VALUES (?,?,?,?,?,?)",
            (review_id, decision.value, final_reply, manager_name, comment, datetime.now().isoformat(timespec="seconds")))
        self.connection.commit()

    def query(self, sql: str) -> pd.DataFrame:
        return pd.read_sql_query(sql, self.connection)

repository = ReviewRepository()
print("✅ SQLite DB sukurta:", repository.db_path)

✅ SQLite DB sukurta: reviewinsight.db


## Visas konvejeris (`ReviewPipeline`) – nuo failo iki DI rezultatų

Paeiliui iškviečiami visi moduliai ir grąžinama apdorojimo santrauka.

In [14]:
#@title 🚀 ReviewPipeline – paleidimas
class ReviewPipeline:
    def __init__(self, loader, preprocessor, analyzer, repository, request_delay: float = 5.0):
        self.loader, self.preprocessor = loader, preprocessor
        self.analyzer, self.repository = analyzer, repository
        self.request_delay = request_delay   # 🆕 pauzė (s) tarp Gemini užklausų – nemokamo plano RPM limitui
        self.results: dict = {}

    def run(self, file_path: str) -> dict:
        reviews = self.loader.to_reviews(self.loader.load_file(file_path))
        reviews = self.preprocessor.process(reviews)
        self.repository.save_reviews(reviews)
        summary = {"įkelta": len(reviews), "atmesta": 0, "išanalizuota": 0, "klaidos": 0}
        first_request = True                     # 🆕
        for r in reviews:
            if r.status == ReviewStatus.REJECTED:
                summary["atmesta"] += 1
                print(f"⏭️  {r.review_id}: atmestas ({r.reject_reason}) – Gemini nekviečiamas")
                continue
            if not first_request:                # 🆕 pauzė prieš kiekvieną, išskyrus pirmą, užklausą
                time.sleep(self.request_delay)
            first_request = False
            try:
                result = self.analyzer.analyze(r)
                self.repository.save_analysis(result)
                self.repository.update_status(r.review_id, ReviewStatus.ANALYZED)
                self.results[r.review_id] = result
                summary["išanalizuota"] += 1
                print(f"✅ {r.review_id}: {result.sentiment:<8} | {result.urgency:<8} | "
                      f"{', '.join(result.categories)}  (bandymų: {result.attempts}, modelis: {result.model_name})")
            except AnalysisError as e:
                self.repository.update_status(r.review_id, ReviewStatus.ANALYSIS_FAILED)
                summary["klaidos"] += 1
                print(f"❌ {e}")
        return summary

pipeline = ReviewPipeline(loader, ReviewPreprocessor(), analyzer, repository, request_delay=5)
t0 = time.time()
run_summary = pipeline.run(INPUT_PATH)
print(f"\n📊 Santrauka: {run_summary}  |  trukmė: {time.time() - t0:.1f} s")

❌ R001: nepavyko po 3 bandymų – API klaida: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED
❌ R002: nepavyko po 3 bandymų – API klaida: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED
❌ R003: nepavyko po 3 bandymų – API klaida: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more info
❌ R004: nepavyko po 3 bandymų – API klaida: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more info
❌ R005: nepavyko po 3 bandymų – API klaida: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED
⏭️  R006: atmestas (per t

In [ ]:
#@title 🔍 Vieno atsiliepimo kelias: įvestis X → prompt → Gemini JSON → išvestis
example_id = "R004"
r = next(x for x in loader.to_reviews(df_input) if x.review_id == example_id)
print("━━━ ĮVESTIS (X) ━━━")
print(json.dumps({k: (int(v) if k == "rating" else v) for k, v in asdict(r).items()
                  if k in loader.required_columns}, ensure_ascii=False, indent=2))
print("\n━━━ GEMINI IŠVESTIS (ŷ) ━━━")
res = pipeline.results.get(example_id)
print(json.dumps(asdict(res), ensure_ascii=False, indent=2) if res else "rezultato nėra")
print("\n━━━ LAUKIAMAS REZULTATAS (y) ━━━")
expected = {e["review_id"]: e for e in json.load(open(EXPECTED_PATH, encoding="utf-8"))}
print(json.dumps(expected[example_id], ensure_ascii=False, indent=2))

In [ ]:
#@title 📋 Visų DI rezultatų lentelė (iš SQLite)
df_results = repository.query("""
    SELECT r.review_id, r.rating, r.language, r.status, a.sentiment, a.confidence, a.urgency,
           a.categories, a.summary, a.reply_draft
    FROM reviews r LEFT JOIN analysis a USING(review_id) ORDER BY r.review_id""")
pd.set_option("display.max_colwidth", 120)
df_results

## Žingsnis 6 – suvestinės ir vizualizacija (`ReportGenerator`)

In [ ]:
#@title 📊 ReportGenerator
class ReportGenerator:
    def __init__(self, repository):
        self.repository = repository

    def sentiment_distribution(self) -> pd.Series:
        return self.repository.query("SELECT sentiment FROM analysis")["sentiment"].value_counts()

    def category_distribution(self) -> pd.Series:
        cats = self.repository.query("SELECT categories FROM analysis")["categories"].map(json.loads)
        return cats.explode().value_counts()

    def urgency_distribution(self) -> pd.Series:
        s = self.repository.query("SELECT urgency FROM analysis")["urgency"].value_counts()
        return s.reindex(["low", "medium", "high", "critical"]).fillna(0)

    def plot_dashboard(self):
        fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
        colors = {"positive": "#2E7D32", "neutral": "#9E9E9E", "negative": "#C62828", "mixed": "#F9A825"}
        s = self.sentiment_distribution()
        ax[0].pie(s.values, labels=s.index, autopct="%1.0f%%", colors=[colors[i] for i in s.index], startangle=90)
        ax[0].set_title("Nuotaikų pasiskirstymas")
        c = self.category_distribution().sort_values()
        ax[1].barh(c.index, c.values, color="#3A6EA5"); ax[1].set_title("Problemų kategorijos")
        u = self.urgency_distribution()
        ax[2].bar(u.index, u.values, color=["#66BB6A", "#FFCA28", "#FB8C00", "#C62828"]); ax[2].set_title("Skubumo lygiai")
        from matplotlib.ticker import MaxNLocator
        ax[1].xaxis.set_major_locator(MaxNLocator(integer=True))
        ax[2].yaxis.set_major_locator(MaxNLocator(integer=True))
        plt.suptitle("ReviewInsight AI – atsiliepimų suvestinė", fontsize=14, fontweight="bold")
        plt.tight_layout(); plt.show()

ReportGenerator(repository).plot_dashboard()

## Rezultatų įvertinimas: DI išvestis (ŷ) prieš laukiamą rezultatą (y)

Struktūriniai laukai lyginami automatiškai: `status`, `language`, `sentiment`, `urgency` – tikslus sutapimas;
`categories` – Jaccard panašumas (sankirta / sąjunga). Laisvo teksto laukus (`summary`, `reply_draft`) vertina žmogus peržiūros metu.

In [ ]:
#@title 🎯 Palyginimas su data/y/
def jaccard(a, b):
    a, b = set(a or []), set(b or [])
    return 1.0 if not a and not b else len(a & b) / len(a | b)

status_now = dict(repository.query("SELECT review_id, status FROM reviews").values)
lang_now = dict(repository.query("SELECT review_id, language FROM reviews").values)
rows = []
for rid, y in expected.items():
    yhat = pipeline.results.get(rid)
    rows.append({
        "review_id": rid,
        "status ✓": status_now.get(rid) == y["status"],
        "kalba ✓": (None if pd.isna(lang_now.get(rid)) else lang_now.get(rid)) == y["language"],
        "sentiment (y / ŷ)": f'{y["sentiment"]} / {yhat.sentiment if yhat else None}',
        "sentiment ✓": (yhat.sentiment if yhat else None) == y["sentiment"],
        "urgency (y / ŷ)": f'{y["urgency"]} / {yhat.urgency if yhat else None}',
        "urgency ✓": (yhat.urgency if yhat else None) == y["urgency"],
        "kategorijų Jaccard": round(jaccard(y["categories"], yhat.categories if yhat else []), 2),
    })
df_eval = pd.DataFrame(rows)
display(df_eval)

metrics = {
    "Statusas": df_eval["status ✓"].mean(),
    "Kalba": df_eval["kalba ✓"].mean(),
    "Nuotaika": df_eval["sentiment ✓"].mean(),
    "Skubumas": df_eval["urgency ✓"].mean(),
    "Kategorijos\n(vid. Jaccard)": df_eval["kategorijų Jaccard"].mean(),
}
plt.figure(figsize=(9, 4))
bars = plt.bar(metrics.keys(), [v * 100 for v in metrics.values()], color="#3A6EA5")
plt.bar_label(bars, fmt="%.0f%%"); plt.ylim(0, 110); plt.ylabel("%")
plt.title("DI rezultatų atitikimas laukiamiems (y)"); plt.tight_layout(); plt.show()

In [ ]:
#@title 💾 Rezultatų eksportas
os.makedirs("results", exist_ok=True)
df_results.to_csv("results/analysis_results.csv", index=False)
df_eval.to_csv("results/evaluation.csv", index=False)
json.dump([asdict(a) for a in pipeline.results.values()],
          open("results/analysis_results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
pd.DataFrame(analyzer.log).to_csv("results/error_log.csv", index=False)
print("✅ Išsaugota į results/:", os.listdir("results"))
print("Klaidų žurnalas:", analyzer.log or "tuščias")